# 00. Configs

This notebook is the single source of truth for all pipeline-wide settings.
It is referenced by all downstream notebooks via %run 00. configs.

Contents:
- Required library imports
- SparkSession initialisation
- Schema and table name constants following the Medallion architecture (Bronze, Silver, Gold)
- Source data path
- Business logic constants (HDB lease duration and valid flat types)
- Date range filter boundaries
- Delta table creation

To customise the pipeline, only edit values in this notebook. All other notebooks inherit these values automatically.

## Required Libraries

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

## Source Data Path
Define where the raw HDB CSV files are located.


In [0]:
SOURCE_PATH = "dbfs:/FileStore/hdb_resale_datasets/"

print(f"Source path : {SOURCE_PATH}")

## Databricks - Catalog
- Databricks-specific catalog usage 
- If using a different cloud platform, comment out the below cell.

In [0]:
catalog="raw"
spark.sql(f"USE CATALOG {catalog}")

DataFrame[]

## Medallion Layer Schema and Table Names

- All schema names and table identifiers are defined here so every notebook resolves them from a single location.

- The pipeline follows the Medallion architecture with three layers.

- Bronze holds raw ingested data and cleaned data records that have passed all data quality checks, plus a separate quarantine table for rejected records.

- Silver - holds the transformed records with the resale identifier, and hashed records with the SHA-256 identifier.

- Gold holds the final enriched outputs: datamart   fact & dim and reporting tables.

- The tables are saved using saveAsTable with the fully qualified name schema.table_name.

In [0]:
# Schema names
BRONZE_SCHEMA  = "bronze"
SILVER_SCHEMA  = "silver"
GOLD_SCHEMA    = "gold"

# Table names
TBL_RAW         = "raw_hdb"
TBL_CLEANED     = "cleaned_hdb"
TBL_FAILED      = "failed_hdb"
TBL_TRANSFORMED = "transformed_hdb"
TBL_HASHED      = "hashed_hdb"

# Fully qualified names used for saveAsTable on cloud: schema.table
CLD_RAW         = f"{BRONZE_SCHEMA}.{TBL_RAW}"
CLD_CLEANED     = f"{BRONZE_SCHEMA}.{TBL_CLEANED}"
CLD_FAILED      = f"{BRONZE_SCHEMA}.{TBL_FAILED}"
CLD_TRANSFORMED = f"{SILVER_SCHEMA}.{TBL_TRANSFORMED}"
CLD_HASHED      = f"{SILVER_SCHEMA}.{TBL_HASHED}"


reporting_targets = {
    "dim_date"                  : f"{GOLD_SCHEMA}.dim_date",
    "dim_location"              : f"{GOLD_SCHEMA}.dim_location",
    "dim_flat"                  : f"{GOLD_SCHEMA}.dim_flat",
    "dim_storey"                : f"{GOLD_SCHEMA}.dim_storey",
    "dim_lease"                 : f"{GOLD_SCHEMA}.dim_lease",
    "fact_resale"               : f"{GOLD_SCHEMA}.fact_resale",
    "rpt_price_by_town_flattype": f"{GOLD_SCHEMA}.rpt_price_by_town_flattype",
    "rpt_lease_vs_price"        : f"{GOLD_SCHEMA}.rpt_lease_vs_price",
    "rpt_storey_price_premium"  : f"{GOLD_SCHEMA}.rpt_storey_price_premium",
}



## Date Range Filter

The pipeline scope covers HDB resale transactions from January 2012 to December 2016 as per the requierment.

In [0]:
DATE_START = "2012-01-01"
DATE_END   = "2016-12-31"

print(f"Date filter : {DATE_START}  to  {DATE_END}")

## Business Logic Constants

- HDB_LEASE is set to 99, which is the standard Singapore HDB lease duration in years. It is used in the Silver stage to compute the lease expiry date and the recomputed remaining lease string.

- VALID_FLAT_TYPES is the whitelist of officially recognised HDB flat types. Any record whose flat_type falls outside this list will be flagged as invalid during the validation step in the Silver stage.

In [0]:
HDB_LEASE = 99

VALID_FLAT_TYPES = [
    "1 ROOM", "2 ROOM", "3 ROOM", "4 ROOM",
    "5 ROOM", "EXECUTIVE", "MULTI-GENERATION"
]

print(f"HDB lease   : {HDB_LEASE} years")
print(f"Valid types : {VALID_FLAT_TYPES}")

## Output Delta Table Creation
- Databricks-specific schema and table creation function. 
- If using a different cloud platform, comment out the schema creation line.

In [0]:
def write_layer(df, fqn, label):
    tgt_schema = fqn.split('.')[0]
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {tgt_schema}")
    df.write.mode("overwrite").option("mergeSchema","true").saveAsTable(fqn)
    print(f"{label} written to table : {fqn}")